# Bias Variance Trade-Offs in RL
Julian Hsu
2025-May-01


## Problem Setup
We run an experiment where we decide the proportion p of users to assign to treatment each round. Treatment has a negative effect on reward, so the optimal policy is to assign everyone to control — but the agent does not know this upfront.

Both arms have the **same within-arm noise**. The only source of variance that depends on p is the between-arm term p·(1-p)·(μ̃_T − μ̃_C)², which is maximised at a balanced 50/50 split and vanishes at the corners (p→0 or p→1). High-λ agents are pushed toward corners to eliminate this term.

**Initialization:** The first round draws p uniformly at random from the candidate grid. With no prior data the agent has no basis to prefer either arm.

**Uncertainty via Thompson sampling:** Each round the agent draws expected arm means from their posteriors — N(x̄, σ²/n) under a flat prior — rather than using point estimates. This uncertainty means the agent cannot immediately identify control as optimal; it must accumulate enough data before its beliefs are reliable. A high-λ agent commits aggressively to whichever corner its current posterior sample favours; a low-λ agent hedges more and corrects faster when early beliefs are wrong.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)


## Thompson Sampling for Arm Means

### Why point estimates are not enough

At each round the agent observes the sample mean $\bar{x}_{arm}$ for each arm. This is a noisy estimate of the true mean $\mu_{arm}$. When an arm has been visited infrequently — e.g. only 3 users per round go to treatment — the sample mean can be far from the truth:

$$\text{Var}(\bar{x}) = \frac{\sigma^2}{n} \qquad \Rightarrow \qquad \text{SE}(\bar{x}) = \frac{\sigma}{\sqrt{n}}$$

With $\sigma = 9$ and $n = 15$ (5 rounds at $p = 0.05$): $\text{SE}(\bar{T}) \approx 2.3$. The agent cannot be confident whether treatment or control has the higher expected reward.

Using point estimates ($\bar{T}$, $\bar{C}$) treats the agent as if it *knows* the arm means exactly. That is wrong early on, and it causes the agent to commit to a corner allocation before it has enough evidence.

### The Bayesian posterior

Assume outcomes are normally distributed with **known** within-arm noise $\sigma^2$:

$$Y_i^{(arm)} \sim \mathcal{N}(\mu_{arm},\ \sigma^2)$$

Under a flat (uninformative) prior on $\mu_{arm}$, after observing $n$ outcomes with sample mean $\bar{x}$, the posterior over the true mean is:

$$\mu_{arm} \mid \text{data} \sim \mathcal{N}\!\left(\bar{x},\ \frac{\sigma^2}{n}\right)$$

The posterior mean equals the sample mean $\bar{x}$, and the posterior standard deviation is $\sigma / \sqrt{n}$. This shrinks toward zero as $n$ grows — the agent's belief about the arm mean becomes more precise as more observations arrive.

### Thompson Sampling decision rule

Instead of plugging $\bar{T}$ and $\bar{C}$ directly into the loss, the agent **samples** from the posteriors each round:

$$\tilde{\mu}_T \sim \mathcal{N}\!\left(\bar{T},\ \frac{\sigma_T^2}{n_T}\right), \qquad \tilde{\mu}_C \sim \mathcal{N}\!\left(\bar{C},\ \frac{\sigma_C^2}{n_C}\right)$$

and evaluates the loss using $(\tilde{\mu}_T,\ \tilde{\mu}_C)$ in place of $(\bar{T},\ \bar{C})$.

**What this achieves:**

| Phase | $n$ small | $n$ large |
|---|---|---|
| Posterior width | Wide — $\sigma/\sqrt{n}$ large | Narrow — $\sigma/\sqrt{n} \to 0$ |
| Agent behaviour | Draws may flip the sign of $\tilde{\mu}_T - \tilde{\mu}_C$ → genuine exploration | Draws cluster near true means → exploits the better arm |

The **feedback loop**: the allocation $p$ chosen this round determines $n_T$ and $n_C$ next round, which determines how fast the posteriors tighten. A high-$\lambda$ agent that commits early to treatment will accumulate many treatment observations (tight posterior on $\mu_T \approx -3$) but few control observations (wide posterior on $\mu_C$) — making it slow to discover that control is better.

In [ ]:
def loss(reward, variance, lambda_weight):
    """L(p) = -reward + lambda * variance.  Minimised by the agent each round."""
    return -reward + lambda_weight * variance


def choose_best_allocation(treatment_outcomes, control_outcomes, p_grid, lambda_weight):
    """Grid-search over p, evaluating loss with Thompson-sampled arm means.

    See the 'Thompson Sampling for Arm Means' cell above for the full derivation.

    Posterior used: mu_arm | data ~ N(x_bar, sigma^2 / n)
      - x_bar  = observed sample mean (posterior mean)
      - sigma/sqrt(n) = posterior SD, shrinks as n grows

    The draw mu_t (or mu_c) replaces the point estimate in both the reward
    and the between-arm variance term, so the agent's entire loss landscape
    reflects its current uncertainty about each arm.
    """
    n_t = len(treatment_outcomes)
    n_c = len(control_outcomes)

    # No data yet: random allocation (agent has no basis to prefer either arm)
    if n_t < 1 or n_c < 1:
        return np.random.choice(p_grid)

    T_bar = np.mean(treatment_outcomes)
    C_bar = np.mean(control_outcomes)

    # Sample variances used for the within-arm variance penalty in the loss.
    # These are asymmetric when arm visit counts differ (small n -> noisy estimate).
    var_T = np.var(treatment_outcomes, ddof=1) if n_t > 1 else noise_treat ** 2
    var_C = np.var(control_outcomes,   ddof=1) if n_c > 1 else noise_control ** 2

    # Thompson sampling: draw from posterior N(x_bar, sigma^2 / n)
    # Posterior SD = sigma / sqrt(n) -- large when n is small, shrinks with data
    mu_t = np.random.normal(T_bar, noise_treat  / np.sqrt(n_t))
    mu_c = np.random.normal(C_bar, noise_control / np.sqrt(n_c))

    best_loss_val = float('inf')
    best_p = p_grid[len(p_grid) // 2]
    for p in p_grid:
        # Reward: expected per-user outcome under allocation p
        reward = p * mu_t + (1 - p) * mu_c

        # Pooled outcome variance (law of total variance):
        #   within-arm:  p*var_T + (1-p)*var_C
        #   between-arm: p*(1-p)*(mu_t - mu_c)^2  <- peaks at p=0.5, zero at corners
        variance = (p * var_T + (1 - p) * var_C
                    + p * (1 - p) * (mu_t - mu_c) ** 2)

        cur = loss(reward, variance, lambda_weight)
        if cur < best_loss_val:
            best_loss_val = cur
            best_p = p
    return best_p


In [ ]:
def outputs(lambda_weight=None):
    allocation_history = []
    cumulative_reward_history = []

    treatment_outcomes = []
    control_outcomes = []
    total_reward = 0

    for t in range(T):
        best_p = choose_best_allocation(
            treatment_outcomes, control_outcomes, p_grid, lambda_weight)

        n_treat   = min(max(int(round(n_per_round * best_p)), 1), n_per_round - 1)
        n_control = n_per_round - n_treat
        allocation_history.append(n_treat / n_per_round)

        treat_outcomes_round = true_effect + np.random.normal(0, noise_treat, n_treat)
        cont_outcomes_round  = np.random.normal(0, noise_control, n_control)

        treatment_outcomes.extend(treat_outcomes_round)
        control_outcomes.extend(cont_outcomes_round)

        total_reward += np.sum(treat_outcomes_round) + np.sum(cont_outcomes_round)
        cumulative_reward_history.append(total_reward)

    return {
        "allocations": allocation_history,
        "cumulative_rewards": cumulative_reward_history,
    }


In [4]:
import pandas as pd

In [ ]:
###### Parameters
# The agent NEVER sees these values -- it only observes realized outcomes.
true_effect   = -3     # treatment lowers the outcome; optimal policy is all-control
noise_control = 9      # both arms have equal noise
noise_treat   = 9      # equal noise means only the between-arm term varies with p

n_per_round   = 50     # users assigned each round
T             = 20     # number of rounds
p_grid        = np.linspace(0.05, 0.95, 19)   # candidate treatment proportions


In [ ]:
lambda_values = [0.0, 0.1, 0.5, 1.5]   # variance-aversion weight (0 = pure reward)
n_runs = 50

results_by_lambda = {
    lw: [outputs(lambda_weight=lw) for _ in range(n_runs)]
    for lw in lambda_values
}

def to_df(results, key):
    return pd.DataFrame([run[key] for run in results])

alloc_dfs  = {lw: to_df(results_by_lambda[lw], "allocations")       for lw in lambda_values}
cumrew_dfs = {lw: to_df(results_by_lambda[lw], "cumulative_rewards") for lw in lambda_values}


In [ ]:
rounds = np.arange(1, T + 1)
optimal_cumrew = np.zeros(T)                          # all control: E[control] = 0
worst_cumrew   = n_per_round * rounds * true_effect   # all treatment: E = true_effect per user

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Treatment Allocation ---
ax = axes[0]
for lw in lambda_values:
    m = alloc_dfs[lw].mean()
    s = alloc_dfs[lw].std()
    (line,) = ax.plot(m.values, label=f'λ = {lw}')
    ax.fill_between(range(T), (m - s).values, (m + s).values, alpha=0.15)
ax.axhline(0.5, linestyle='--', color='gray', linewidth=0.8, label='Balanced (0.5)')
ax.set_title('Mean Treatment Allocation Over Time')
ax.set_xlabel('Round')
ax.set_ylabel('Proportion Treated')
ax.legend()

# --- Cumulative Outcome ---
ax = axes[1]
ax.plot(optimal_cumrew, linestyle='--', color='black', linewidth=1.2,
        label='Optimal (all control)')
ax.plot(worst_cumrew,   linestyle=':',  color='red',   linewidth=1.2,
        label='Worst case (all treatment)')
for lw in lambda_values:
    m = cumrew_dfs[lw].mean()
    s = cumrew_dfs[lw].std()
    (line,) = ax.plot(m.values, label=f'λ = {lw}')
    ax.fill_between(range(T), (m - s).values, (m + s).values, alpha=0.15)
ax.set_title('Cumulative Outcome Over Time')
ax.set_xlabel('Round')
ax.set_ylabel('Total Reward Accumulated')
ax.legend()

plt.tight_layout()
plt.show()


## FAQ: Why Doesn't Known Variance Work?

A natural simplification is to replace the noisy sample variances with the true known $\sigma^2$ and use point estimates $(\bar{T}, \bar{C})$ for the arm means. This section shows algebraically why that approach makes $\lambda$ irrelevant.

---

### Setup

With equal known within-arm variance $\sigma^2$ the loss at allocation $p$ is:

$$L(p) = -\bigl[p\bar{T} + (1-p)\bar{C}\bigr] + \lambda\Bigl[\underbrace{p\sigma^2 + (1-p)\sigma^2}_{=\,\sigma^2\ (\text{constant in }p)} + \underbrace{p(1-p)(\bar{T}-\bar{C})^2}_{\text{between-arm term}}\Bigr]$$

$$= -\bigl[p\bar{T} + (1-p)\bar{C}\bigr] + \lambda\sigma^2 + \lambda\,p(1-p)(\bar{T}-\bar{C})^2$$

---

### The loss is concave → the minimum is always at a corner

$$\frac{d^2 L}{dp^2} = \lambda \cdot \frac{d^2}{dp^2}\bigl[p(1-p)\bigr] \cdot (\bar{T}-\bar{C})^2 = -2\lambda(\bar{T}-\bar{C})^2 \leq 0$$

Because the loss is concave in $p$, the agent always prefers the most extreme allocation available — either $p \approx 0$ (all control) or $p \approx 1$ (all treatment). Intermediate splits are never optimal.

---

### λ cancels in the corner comparison

The between-arm term $p(1-p)$ evaluates to the **same value** at both extreme grid points:

$$p = 0.05\!: \quad 0.05 \times 0.95 = 0.0475$$
$$p = 0.95\!: \quad 0.95 \times 0.05 = 0.0475$$

So:

$$L(0.95) = -[0.95\bar{T} + 0.05\bar{C}] + \lambda\sigma^2 + 0.0475\lambda(\bar{T}-\bar{C})^2$$

$$L(0.05) = -[0.05\bar{T} + 0.95\bar{C}] + \lambda\sigma^2 + 0.0475\lambda(\bar{T}-\bar{C})^2$$

$$\boxed{L(0.95) - L(0.05) = 0.9\,(\bar{C} - \bar{T})}$$

Every $\lambda$ term cancels. The agent picks all-treatment when $\bar{T} > \bar{C}$ and all-control when $\bar{C} > \bar{T}$ — **the same corner for every value of $\lambda$**. Varying $\lambda$ from 0 to 100 produces identical allocation decisions.

---

### Why Thompson Sampling restores the λ effect

Thompson Sampling introduces two asymmetries that known variance erases:

1. **Asymmetric sample variances.** When the agent spends many rounds near $p = 0.05$, the treatment arm accumulates few observations ($n_T$ small). Its sample variance `var_T` is estimated from a handful of data points and is often far from $\sigma^2$. High-$\lambda$ agents amplify this discrepancy: if `var_T` happens to look small (cheap to allocate to), they commit hard to treatment even when $\bar{T} < \bar{C}$.

2. **Stochastic mean draws.** The TS draw $\tilde{\mu}_T \sim \mathcal{N}(\bar{T},\, \sigma^2/n_T)$ preserves uncertainty even after observing data. A high-$\lambda$ agent that sees a lucky draw $\tilde{\mu}_T > \tilde{\mu}_C$ will commit aggressively to the treatment corner and be slow to self-correct, because it then collects little control data and the posterior on $\mu_C$ stays wide.

Both effects are proportional to $\lambda$ and disappear when variances are known and equal.